In [35]:
import pandas as pd
from IPython.display import display


In [4]:
df = pd.read_csv('tintos\caracteristicas_tintos_merged_150_to_202.csv')
df.head

<bound method NDFrame.head of             ID  Ligero/Poderoso  Suave/Tánico  Seco/Dulce  Débil/Ácido
0      9716238              4.2           6.0         0.0          5.2
1      1944452              5.2           7.3         0.0          8.0
2      1214057              6.6           5.7         2.5          3.9
3      1596006              4.9           5.4         0.1          6.2
4      6000319              6.5           6.4         0.0          5.4
5      2153344              7.6           3.2         0.8          3.9
6      1151148              6.2           5.7         0.8          5.6
7      5616661              3.9           2.1         0.0          7.1
8      1462973              6.7           5.6         1.3          6.2
9      1754937              6.3           3.8         0.4          4.4
10     1110325              7.3           5.5         1.2          3.1
11     9998471              2.6           1.7         0.0          5.7
12     1143056              8.0           2.0  

In [5]:
df.columns

Index(['ID', 'Ligero/Poderoso', 'Suave/Tánico', 'Seco/Dulce', 'Débil/Ácido'], dtype='object')

In [9]:
df.duplicated().sum()

np.int64(121)

In [14]:
df.shape

(5220, 5)

In [13]:
# Mostrar todas las filas duplicadas (tanto la primera como las duplicadas)
duplicates = df[df.duplicated(keep=False)]

# Ordenar para que las filas duplicadas estén juntas
duplicates_sorted = duplicates.sort_values(by=list(df.columns))

# Mostrar las filas duplicadas
print(duplicates_sorted)


            ID  Ligero/Poderoso  Suave/Tánico  Seco/Dulce  Débil/Ácido
2260      9400              7.6           5.9         0.5          3.0
2335      9400              7.6           5.9         0.5          3.0
2309     19472              5.5           2.5         1.7          3.4
2380     19472              5.5           2.5         1.7          3.4
2290     22823              5.8           3.0         0.2          2.9
2361     22823              5.8           3.0         0.2          2.9
661      27324              7.3           3.8         2.2          3.7
711      27324              7.3           3.8         2.2          3.7
660      74629              7.6           6.8         0.3          5.6
710      74629              7.6           6.8         0.3          5.6
2284     82652              7.5           3.7         2.7          2.4
2355     82652              7.5           3.7         2.7          2.4
2308     85749              6.4           4.2         0.3          4.1
2379  

In [15]:
df = df.drop_duplicates()

In [16]:
df.shape

(5099, 5)

In [17]:
# Guardar el DataFrame limpio en un archivo CSV
df.to_csv(r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\caracteristicas_tintos_merged_150_to_202.csv', index=False, encoding='utf-8')

print("El archivo CSV se ha guardado correctamente.")


El archivo CSV se ha guardado correctamente.


In [22]:
import pandas as pd

# Leer los dos archivos CSV
df1 = pd.read_csv(r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\DF_BF_tintos_150to202.csv', usecols=['Url', 'ID'])
df2 = pd.read_csv(r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\caracteristicas_tintos_merged_150_to_202.csv', usecols=['ID'])

# Encontrar los IDs que están en df1 pero no en df2
ids_not_in_df2 = df1[~df1['ID'].isin(df2['ID'])]

# Ruta del archivo donde se guardarán los resultados
output_file = r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\ids_no_en_segundo_csv.txt'

# Guardar las URLs y IDs en un archivo de texto
with open(output_file, 'w', encoding='utf-8') as file:
    for index, row in ids_not_in_df2.iterrows():
        file.write(f"URL: {row['Url']}\n")

print(f"Los resultados se han guardado en {output_file}")



Los resultados se han guardado en C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\ids_no_en_segundo_csv.txt


In [23]:
import pandas as pd

# Ruta del archivo .txt
input_file = r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\ids_no_en_segundo_csv.txt'

# Leer el archivo .txt, asumiendo que cada URL está en una línea
with open(input_file, 'r', encoding='utf-8') as file:
    # Leer las líneas y eliminar el prefijo "URL: " de cada una
    urls = [line.strip().replace('URL: ', '') for line in file.readlines()]

# Crear un DataFrame con la lista de URLs
df_urls = pd.DataFrame(urls, columns=['URL'])

# Ruta de salida para el archivo CSV
output_file = r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\urls_limpios.csv'

# Guardar el DataFrame limpio en un archivo CSV
df_urls.to_csv(output_file, index=False, encoding='utf-8')

print(f"El archivo CSV con las URLs se ha guardado en {output_file}")


El archivo CSV con las URLs se ha guardado en C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\urls_limpios.csv


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import random
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
import csv
import requests
import pandas as pd
import re

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import os


# Leer los enlaces desde un archivo de texto
with open(r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\def_tintos_merged_150_to_202.txt', 'r', encoding='utf-8-sig') as f:
    urls = f.readlines()  # Lee todos los enlaces en el archivo

# Limitar a los primeros 5 enlaces
#urls = urls[:2]

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar una lista para guardar todos los datos
all_wine_data = []

# Recorrer cada URL en la lista
for url in urls:
    url = url.strip()  # Eliminar cualquier espacio en blanco o salto de línea

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Inicializar datos
        wine_data = {}

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        #grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'


        #Grape
        grapes = soup.find_all("a", class_="anchor_anchor__m8Qi- wineFacts__link--3aTg9")
        grape = [grape.text.strip() for grape in grapes if "grapes" in grape["href"]]
        grape = ', '.join(grape) if grape else 'No disponible'


        # Nombre del vino 
        wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
        if wine_headline:
            # Tomamos todo el texto dentro del bloque, sin intentar separarlo
            name = wine_headline.get_text(strip=True)
        else:
            name = 'No disponible'
        
            # Si el nombre del vino contiene el nombre de la bodega, eliminamos la bodega del nombre
        if winery.lower() in name.lower():
            name = name.replace(winery, '').strip()


        # Año
        button_elements = soup.find_all('button', class_='MuiButtonBase-root')
        year = 'No disponible'

            # Buscar en los botones primero
        for button in button_elements:
            if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
                year = button.get('aria-label').strip()
                break

            # Si no se encuentra en los botones, buscar en el span con la clase 'vivino-mui-14ngluw-componentChildren'
        if year == 'No disponible':
            year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
            if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
                year = year_element.text.strip()

            # Si aún no se ha encontrado, buscar todos los 'span' con la clase 'vintageListRow__year--34Tuc' y tomar el primero
        if year == 'No disponible':
            vintage_section = soup.find('div', id='vintageListSection')
            if vintage_section:
                # Buscar todos los 'span' dentro del div y filtrar aquellos que contienen un año (4 dígitos)
                year_elements = vintage_section.find_all('span', string=re.compile(r'\d{4}'))
                if year_elements:
                    year = year_elements[0].string.strip()  # Usamos .string para obtener solo el texto


        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        if price_element:
            price = price_element.text.replace('€', '').replace('\xa0', '').strip()
        else:
            # Si no lo encuentra, busca el precio en la segunda clase
            price_element = soup.find(class_='purchaseAvailabilityPPC__amount--2_4GT')
            if price_element:
                # Utilizamos regex para encontrar el precio con la coma
                match = re.search(r'\d{1,3}(?:,\d{3})*(?:\.\d+)?', price_element.text)
                if match:
                    price = match.group(0).replace('\xa0', '').strip()
                else:
                    price = 'No disponible'
            else:
                price = 'No disponible'

        
        # Grados de Alcohol
        alcohol_element = soup.find(class_='wineFacts__wineFacts--2Ih8B')
            # Buscar todos los spans dentro de la tabla
        if alcohol_element:
            spans = alcohol_element.find_all('span')
            # Filtrar los spans que contienen un número seguido de '%' (grado de alcohol)
        alcohol = 'No disponible'
        for span in spans:
                # Usamos una expresión regular para buscar un número seguido de '%'
                match = re.search(r'\d+%', span.text.strip())
                if match:
                    alcohol = match.group(0)  # El valor que coincide con la expresión regular
                    alcohol = alcohol.replace('%', '')
                    break  # Detener la búsqueda cuando encontramos el primer grado de alcohol
        else:
            alcohol = 'No disponible'



        # Notas de sabor
        taste_containers = soup.find_all(class_='slider__viewPort--30MrB')
        taste_notes = []
        for container in taste_containers:
            # Encontrar todos los elementos con la clase 'tasteNote__popularKeywords--1gIa2'
            taste_keywords = container.find_all(class_='tasteNote__popularKeywords--1gIa2')

            # Recorrer cada uno de los elementos encontrados y extraer el texto
            for keyword in taste_keywords:
                if keyword.text.strip():  # Solo si no está vacío
                    taste_notes.append(keyword.text.strip())

        # Unir todas las notas en una sola cadena, separada por coma
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        
        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]


        # Guardar los datos de esta URL
        wine_data = {
            'Url': url,
            'ID': re.search(r'\/(\d+)$', url).group(1),
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Contenido de alcohol': alcohol,
            'Maridajes':', '.join(pairings),
            
        }
        all_wine_data.append(wine_data)

    except Exception as e:
        print(f"Error al procesar la URL {url}: {e}")

# Guardar los resultados en un archivo CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir los datos a un DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)  # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

print(df.head())


Error al procesar la URL --- def_tintos150.txt ---: Invalid URL '--- def_tintos150.txt ---': No scheme supplied. Perhaps you meant https://--- def_tintos150.txt ---?
Error al procesar la URL : Invalid URL '': No scheme supplied. Perhaps you meant https://?
Error al procesar la URL : Invalid URL '': No scheme supplied. Perhaps you meant https://?
Error al procesar la URL --- def_tintos151.txt ---: Invalid URL '--- def_tintos151.txt ---': No scheme supplied. Perhaps you meant https://--- def_tintos151.txt ---?
Error al procesar la URL : Invalid URL '': No scheme supplied. Perhaps you meant https://?
Error al procesar la URL : Invalid URL '': No scheme supplied. Perhaps you meant https://?
Error al procesar la URL --- def_tintos152.txt ---: Invalid URL '--- def_tintos152.txt ---': No scheme supplied. Perhaps you meant https://--- def_tintos152.txt ---?
Error al procesar la URL : Invalid URL '': No scheme supplied. Perhaps you meant https://?
Error al procesar la URL : Invalid URL '': No s